# 01 · Escada de equilibração — minimização, aquecimento e restrições

| | |
|---|---|
| **Autor** | Patrick Allan dos Santos Faustino |
| **Projeto** | Revisão tutorial de dinâmica molecular — xilanase GH11 (PDB 1XNB) |
| **Versão** | v2026.11 — notebook autocontido |

**Pergunta.** As minimizações relaxaram o sistema, o aquecimento foi gradual e as restrições posicionais caíram progressivamente, como o protocolo pede?

Documenta o **protocolo**, não a proteína: etapas em sequência, no espírito do protocolo de dez
passos de Roe & Brooks (*J. Chem. Phys.* **2020**, 153, 054123).

* **A — minimizações:** energia potencial contra o passo acumulado, com o desfecho lido do `.log`.
  O termo `Potential` do GROMACS **inclui** a energia das restrições posicionais (`Position Rest.`):
  com `DESCONTAR_RESTRICOES = True` ela é subtraída, para que etapas com k diferentes sejam
  comparáveis. `ESCALA_ENERGIA = "log"` mostra E − E_final de cada etapa, a forma de ver a
  convergência quando a energia inicial é muito alta (contatos ruins).
* **B — dinâmicas:** temperatura contra o tempo acumulado, com a rampa nominal do `annealing`.
* **C — restrições:** constante de força de cadeia principal e laterais, etapa a etapa.

> **Minimização não tem tempo.** Num `.edr` de `steep` o campo `Time` é o número do passo: o
> painel A tem eixo de **passos**, o B de **tempo**.
>
> As constantes de força são **declaradas** em `ETAPAS` (o `.edr` guarda a energia das restrições,
> não o `POSRES_FC`): copie-as das linhas `define` dos `.mdp`. O painel C é a conferência visual.

**Blocos:** 1 bibliotecas · 2 configuração e dados · 3 cálculos · 4 estatística ·
5 gráfico (matplotlib) · 6 gráfico interativo (Plotly) · 7 exportação.

## 1 · Bibliotecas

In [ ]:
import sys
import platform
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re
import panedr
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

INICIO = datetime.now()
VERSOES = {
    "Python": platform.python_version(),
    "NumPy": np.__version__,
    "pandas": pd.__version__,
    "SciPy": scipy.__version__,
    "Matplotlib": matplotlib.__version__,
    "Plotly": plotly.__version__,
    "panedr": panedr.__version__,
}
print(f"Data e hora da execução: {INICIO:%d/%m/%Y %H:%M:%S}")
print(f"Sistema operacional: {platform.platform()}")
df_versoes = pd.DataFrame(list(VERSOES.items()), columns=["biblioteca", "versao"])
df_versoes

## 2 · Configuração da análise e carregamento dos dados

In [ ]:
# ── Onde estão os arquivos ──────────────────────────────────────────────────
PASTA_BASE = Path("..")             # <- AJUSTE
SISTEMA = "xilanase_1XNB"
REPLICAS = "protocolo (uma réplica)"

# ── Etapas, em ORDEM de execução ────────────────────────────────────────────
# tipo: "minimizacao" ou "dinamica"; fc_*: kJ mol⁻¹ nm⁻² (0 = livre); caminhos relativos a PASTA_BASE
ETAPAS = [
    {"rotulo": "EM1",  "arquivo": "em1.edr",      "tipo": "minimizacao", "fc_bb": 2092.0, "fc_sc": 2092.0},
    {"rotulo": "EM2",  "arquivo": "em2.edr",      "tipo": "minimizacao", "fc_bb":  836.8, "fc_sc":  836.8},
    {"rotulo": "EM3",  "arquivo": "em3.edr",      "tipo": "minimizacao", "fc_bb":   41.84, "fc_sc":  41.84},
    {"rotulo": "EM4",  "arquivo": "em4.edr",      "tipo": "minimizacao", "fc_bb":    0.0, "fc_sc":    0.0},
    {"rotulo": "NVT1", "arquivo": "rep1/nvt1.edr", "tipo": "dinamica",   "fc_bb": 2092.0, "fc_sc": 2092.0},
    {"rotulo": "NVT2", "arquivo": "rep1/nvt2.edr", "tipo": "dinamica",   "fc_bb":  836.8, "fc_sc":  836.8},
    {"rotulo": "NPT1", "arquivo": "rep1/npt1.edr", "tipo": "dinamica",   "fc_bb":  418.4, "fc_sc":  418.4},
    {"rotulo": "NPT2", "arquivo": "rep1/npt2.edr", "tipo": "dinamica",   "fc_bb":  209.2, "fc_sc":  209.2},
    {"rotulo": "NPT3", "arquivo": "rep1/npt3.edr", "tipo": "dinamica",   "fc_bb":  209.2, "fc_sc":    0.0},
    {"rotulo": "NPT4", "arquivo": "rep1/npt4.edr", "tipo": "dinamica",   "fc_bb":    0.0, "fc_sc":    0.0},
]

# ── Análise ─────────────────────────────────────────────────────────────────
TERMO_MINIMIZACAO = "Potential"       # painel A
DESCONTAR_RESTRICOES = True           # E_pot − "Position Rest.": compara etapas com k diferentes
TERMO_DINAMICA = "Temperature"        # painel B
TEMPERATURA_ALVO = 298.15
ETAPA_AQUECIMENTO = "NVT1"            # rampa do .mdp (annealing-time em ps, a partir do início da etapa)
ANNEALING = {"tempo_ps": [0.0, 400.0], "temperatura_K": [50.0, 298.15]}   # None = sem rampa
FRACAO_FINAL = 0.2                    # último quinto da etapa = trecho "relaxado"
NIVEL_CONFIANCA = 0.95
LER_LOGS = True                       # lê o .log de mesmo nome das minimizações

DIR_SAIDA = Path("resultados") / SISTEMA / "01_etapas_equilibracao"
PREFIXO = "etapas"
TITULO = "01 · Escada de equilibração"

In [ ]:
df_arquivos = pd.DataFrame([{"etapa": e["rotulo"], "tipo": e["tipo"], "caminho": str(PASTA_BASE / e["arquivo"]),
                             "existe": (PASTA_BASE / e["arquivo"]).exists(),
                             "log": (PASTA_BASE / e["arquivo"]).with_suffix(".log").exists()} for e in ETAPAS])
display(df_arquivos)
if not df_arquivos["existe"].all():
    raise FileNotFoundError("Arquivos ausentes (veja a tabela acima).")

In [ ]:
def ler_energia(caminho):
    """.edr (panedr) ou .xvg (gmx energy) -> DataFrame com a coluna Time em ps.

    O panedr devolve Time como coluna E como índice: um reset_index cego duplicaria a coluna.
    """
    caminho = str(caminho)
    if caminho.endswith(".edr"):
        dados = panedr.edr_to_df(caminho)
        if "Time" not in dados.columns:
            dados = dados.reset_index().rename(columns={"index": "Time"})
        dados = dados.reset_index(drop=True)
        return dados.loc[:, ~dados.columns.duplicated()]
    linhas = open(caminho).read().splitlines()
    legendas = [l.split('"')[1] for l in linhas if l.startswith("@ s") and '"' in l]
    matriz = np.array([[float(v) for v in l.split()] for l in linhas
                       if l.strip() and not l.startswith(("#", "@"))])
    return pd.DataFrame(matriz, columns=["Time"] + legendas[:matriz.shape[1] - 1])


def coluna(dados, termo):
    """Nome da coluna do termo: exato primeiro, depois trecho do nome."""
    if termo in dados.columns:
        return termo
    achados = [c for c in dados.columns if termo.lower() in c.lower()]
    if not achados:
        raise ValueError(f"'{termo}' não está no arquivo. Termos: {list(dados.columns)}")
    return achados[0]



def passos_do_log(arquivo_edr):
    """Do .log do `steep`: passos executados, desfecho, Fmax e Epot finais.

    Três desfechos do GROMACS: 'converged to Fmax' (critério atingido), 'converged to machine
    precision' (parou no limite numérico SEM atingi-lo) e 'did not converge' (nsteps esgotado).
    """
    caminho = Path(arquivo_edr).with_suffix(".log")
    if not caminho.exists():
        return {"passos_executados": np.nan, "desfecho": "log ausente", "fmax_final": np.nan, "epot_final": np.nan}
    texto = caminho.read_text(errors="ignore")
    desfecho = ("precisão de máquina (Fmax NÃO atingido)" if "converged to machine precision" in texto
                else "Fmax atingido" if "converged to Fmax" in texto
                else "nsteps esgotado (Fmax NÃO atingido)" if "did not converge" in texto else "não identificado")
    ultimo = lambda padrao, tipo: (tipo(re.findall(padrao, texto)[-1]) if re.findall(padrao, texto) else np.nan)
    return {"passos_executados": ultimo(r"converge[d]?\b.*?\bin\s+(\d+)\s+steps", int), "desfecho": desfecho,
            "fmax_final": ultimo(r"Maximum force\s*=\s*([-+0-9.eE]+)", float),
            "epot_final": ultimo(r"Potential Energy\s*=\s*([-+0-9.eE]+)", float)}


energias, linhas = {}, []
for e in ETAPAS:
    dados = ler_energia(PASTA_BASE / e["arquivo"])
    termo = coluna(dados, TERMO_MINIMIZACAO if e["tipo"] == "minimizacao" else TERMO_DINAMICA)
    valores = dados[termo].to_numpy(dtype=float)
    if e["tipo"] == "minimizacao" and DESCONTAR_RESTRICOES and "Position Rest." in dados.columns:
        valores = valores - dados["Position Rest."].to_numpy(dtype=float)   # o Potential do GROMACS inclui as restrições
        termo = f"{termo} − Position Rest."
    energias[e["rotulo"]] = (dados["Time"].to_numpy(dtype=float), valores, termo)
    linha = {"etapa": e["rotulo"], "tipo": e["tipo"], "termo": termo, "pontos": len(dados),
             "eixo_final": dados["Time"].iloc[-1], "fc_bb": e["fc_bb"], "fc_sc": e["fc_sc"]}
    if e["tipo"] == "minimizacao" and LER_LOGS:
        linha.update(passos_do_log(PASTA_BASE / e["arquivo"]))
    linhas.append(linha)
df_leitura = pd.DataFrame(linhas)
df_leitura

## 3 · Cálculos

As etapas são concatenadas **dentro de cada tipo**: passos acumulados nas minimizações, tempo
acumulado nas dinâmicas. Um intervalo de amostragem separa uma etapa da seguinte.

In [ ]:
quadros, deslocamento = [], {"minimizacao": 0.0, "dinamica": 0.0}
for e in ETAPAS:
    eixo, valor, termo = energias[e["rotulo"]]
    local = eixo - eixo[0]
    acumulado = local + deslocamento[e["tipo"]]
    quadros.append(pd.DataFrame({"etapa": e["rotulo"], "tipo": e["tipo"], "termo": termo, "eixo_local": local,
                                 "eixo_acumulado": acumulado, "valor": valor, "fc_bb": e["fc_bb"], "fc_sc": e["fc_sc"]}))
    deslocamento[e["tipo"]] = acumulado[-1] + (float(np.median(np.diff(eixo))) if eixo.size > 1 else 1.0)
df_series = pd.concat(quadros, ignore_index=True)
df_series["tempo_acumulado_ns"] = np.where(df_series["tipo"] == "dinamica", df_series["eixo_acumulado"] / 1000.0, np.nan)
print(f"minimização: {deslocamento['minimizacao']:.0f} passos | dinâmica: {deslocamento['dinamica'] / 1000:.2f} ns")
df_series.groupby(["tipo", "etapa"], sort=False)["valor"].describe()

## 4 · Estatística

Por etapa: valores inicial e final e, no **último quinto** (o trecho já relaxado), média, DP e
SEM corrigido pela ineficiência estatística (Chodera *et al.*, *JCTC* **2007**, 3, 26). Nas
dinâmicas, o desvio do alvo de temperatura: se a última etapa não fecha no alvo, o termostato
não acompanhou a rampa (`tau-t` alto) ou `ref-t` está errado. A escada de restrições é conferida
**por bloco** (a minimização termina livre e a dinâmica recomeça restringida, por desenho).

In [ ]:
def ineficiencia_estatistica(x, t_minimo=3):
    """g = 1 + 2·τ_int (em frames) — Chodera et al., JCTC 2007, 3, 26.

    ACF não enviesada (FFT), somada até o 1º C(t) <= 0 depois de t_minimo, com o
    peso (1 − t/N) aplicado UMA vez. Igual a
    pymbar.timeseries.statistical_inefficiency(fast=False).
    """
    x = np.asarray(x, dtype=float)
    n = x.size
    if n < 3 or np.allclose(x, x[0]):
        return 1.0
    d = x - x.mean()
    var = float(d @ d) / n
    tamanho = 1 << (2 * n - 1).bit_length()
    f = np.fft.rfft(d, tamanho)
    acf = np.fft.irfft(f * np.conjugate(f), tamanho)[:n] / ((n - np.arange(n)) * var)
    g = 1.0
    for t in range(1, n - 1):
        if acf[t] <= 0.0 and t > t_minimo:
            break
        g += 2.0 * acf[t] * (1.0 - t / n)
    return float(max(g, 1.0))


def t_student(graus_liberdade):
    return stats.t.ppf(0.5 + NIVEL_CONFIANCA / 2, df=max(graus_liberdade, 1))


def resumo_replica(y, dt_ns, observavel, replica, unidade=""):
    """Média, DP, g, N_ef e SEM corrigido de UMA série (uma réplica)."""
    y = np.asarray(y, dtype=float)
    g = ineficiencia_estatistica(y)
    n_ef = max(y.size / g, 2.0)
    dp = float(np.std(y, ddof=1))
    sem = dp / np.sqrt(n_ef)
    media = float(np.mean(y))
    return {"observavel": observavel, "replica": replica, "unidade": unidade,
            "n": y.size, "media": media, "desvio_padrao": dp, "sem": sem,
            "ic_inferior": media - t_student(n_ef - 1) * sem,
            "ic_superior": media + t_student(n_ef - 1) * sem,
            "minimo": float(y.min()), "maximo": float(y.max()),
            "g": g, "tau_int_ns": (g - 1.0) / 2.0 * dt_ns, "n_efetivo": n_ef}


def entre_replicas(medias, observavel, unidade=""):
    """A réplica é a unidade estatística: média ± SEM das médias das réplicas."""
    medias = np.asarray(medias, dtype=float)
    k = medias.size
    media = float(medias.mean())
    dp = float(np.std(medias, ddof=1)) if k > 1 else np.nan
    sem = dp / np.sqrt(k) if k > 1 else np.nan
    return {"observavel": observavel, "replica": "entre_replicas", "unidade": unidade,
            "n": k, "media": media, "desvio_padrao": dp, "sem": sem,
            "ic_inferior": media - t_student(k - 1) * sem if k > 1 else np.nan,
            "ic_superior": media + t_student(k - 1) * sem if k > 1 else np.nan,
            "minimo": float(medias.min()), "maximo": float(medias.max()),
            "g": np.nan, "tau_int_ns": np.nan, "n_efetivo": float(k)}


def estatisticas_series(df_series, unidades):
    """Por réplica (só produção) + linha 'entre_replicas' quando há 2 ou mais."""
    linhas = []
    prod = df_series[df_series["fase"] == "producao"]
    for obs, bloco in prod.groupby("observavel", sort=False):
        por_rep = [resumo_replica(g["valor"], float(np.median(np.diff(g["tempo_ns"]))),
                                  obs, rep, unidades.get(obs, ""))
                   for rep, g in bloco.groupby("replica", sort=False)]
        linhas += por_rep
        if len(por_rep) > 1:
            linhas.append(entre_replicas([l["media"] for l in por_rep], obs, unidades.get(obs, "")))
    return pd.DataFrame(linhas)


def referencia(df_stats, observavel):
    """Linha 'entre_replicas' do observável (ou a da única réplica)."""
    bloco = df_stats[df_stats["observavel"] == observavel]
    entre = bloco[bloco["replica"] == "entre_replicas"]
    return (entre if len(entre) else bloco).iloc[0]


def formatar(media, erro):
    """'média ± erro' com 2 algarismos significativos no erro (GUM, JCGM 100:2008 §7.2.6)."""
    if not (np.isfinite(erro) and erro > 0):
        return f"{media:.4g}"
    casas = max(0, 1 - int(np.floor(np.log10(abs(erro)))))
    return f"{media:.{casas}f} ± {erro:.{casas}f}"


_SOBRESCRITO = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹", "0123456789")


def unidade_mpl(unidade):
    """Unidade para o matplotlib: '⁻³' vira mathtext ($^{-3}$). O sinal sobrescrito '⁻' (U+207B)
    falta em várias fontes (a Aptos Serif, inclusive) e sai como um quadrado na figura."""
    return re.sub(r"⁻([⁰¹²³⁴⁵⁶⁷⁸⁹]+)",
                  lambda m: "$^{-" + m.group(1).translate(_SOBRESCRITO) + "}$", unidade or "")


def rotulo_media(ref):
    """Rótulo dos painéis: 'média ± EP (N réplicas): valor ± erro unidade'.

    EP = erro padrão da média: entre as médias das réplicas (a réplica é a unidade estatística);
    com uma réplica só, o SEM da série corrigido pela ineficiência estatística."""
    n = ref.get("n", np.nan)
    origem = (f"{int(n)} réplicas" if ref.get("replica") == "entre_replicas" and pd.notna(n)
              else "1 réplica")
    return (f"média ± EP ({origem}): {formatar(ref['media'], ref['sem'])} "
            f"{unidade_mpl(ref.get('unidade', ''))}").strip()

In [ ]:
linhas = []
for e in ETAPAS:
    y = df_series.loc[df_series["etapa"] == e["rotulo"], "valor"].to_numpy()
    final = y[-max(2, int(FRACAO_FINAL * y.size)):]
    g = ineficiencia_estatistica(final)
    linha = {"etapa": e["rotulo"], "tipo": e["tipo"], "unidade": "kJ mol⁻¹" if e["tipo"] == "minimizacao" else "K",
             "valor_inicial": y[0], "valor_final": y[-1], "media_ultimo_quinto": final.mean(),
             "desvio_padrao_ultimo_quinto": final.std(ddof=1), "g": g,
             "sem_ultimo_quinto": final.std(ddof=1) / np.sqrt(max(final.size / g, 2.0)),
             "fc_bb": e["fc_bb"], "fc_sc": e["fc_sc"]}
    if e["tipo"] == "minimizacao":
        linha["variacao_energia"] = y[-1] - y[0]
    else:
        linha["desvio_do_alvo_K"] = final.mean() - TEMPERATURA_ALVO
    linhas.append(linha)
df_resumo = pd.DataFrame(linhas)
if LER_LOGS:
    df_resumo = df_resumo.merge(df_leitura[["etapa"] + [c for c in ("passos_executados", "desfecho", "fmax_final")
                                                        if c in df_leitura]], on="etapa", how="left")

for tipo, bloco in df_resumo.groupby("tipo", sort=False):
    for c in ("fc_bb", "fc_sc"):
        if np.any(np.diff(bloco[c].to_numpy()) > 0):
            print(f"[!] '{c}' AUMENTA dentro do bloco de {tipo}: confira as linhas `define` dos .mdp.")
din = df_resumo[df_resumo["tipo"] == "dinamica"]
if len(din) and abs(din["desvio_do_alvo_K"].iloc[-1]) > 1.0:
    print(f"[!] A última dinâmica fecha {din['desvio_do_alvo_K'].iloc[-1]:+.2f} K fora do alvo.")
if "desfecho" in df_resumo and (df_resumo["desfecho"].fillna("").str.contains("NÃO")).any():
    print("[!] Alguma minimização terminou sem atingir Fmax — relate o desfecho, não só 'minimizado'.")
df_resumo.assign(**{"último quinto (média ± SEM)": [formatar(m, s) for m, s in
                                                     zip(df_resumo["media_ultimo_quinto"], df_resumo["sem_ultimo_quinto"])]})

## 5 · Gráfico (matplotlib)

In [ ]:
# ── Estilo geral (vale para todas as figuras deste notebook) ────────────────
FONTE = "Arial"                 # se não estiver instalada, usa DejaVu Serif
TAMANHO_FONTE = 10                     # pt
CORES = ["#003f5c", "#bc5090", "#ffa600", "#58508d", "#ff6361"]   # uma por réplica
COR_MEDIA = "#333333"                 # tracejado da média
COR_REFERENCIA = "#d62728"            # linhas de referência e faixa do descarte

if FONTE not in {f.name for f in font_manager.fontManager.ttflist}:
    FONTE = "DejaVu Serif"
plt.rcParams.update({
    "font.family": FONTE, "font.size": TAMANHO_FONTE,
    "axes.labelsize": TAMANHO_FONTE, "axes.titlesize": TAMANHO_FONTE,
    "xtick.labelsize": TAMANHO_FONTE - 1, "ytick.labelsize": TAMANHO_FONTE - 1,
    "legend.fontsize": TAMANHO_FONTE - 1, "axes.linewidth": 0.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "figure.dpi": 120,
    "pdf.fonttype": 42,                # fontes embutidas como TrueType (editáveis)
    "svg.fonttype": "none",            # texto do SVG continua texto
})

In [ ]:
# ── Layout desta figura ─────────────────────────────────────────────────────
LARGURA, ALTURA_PAINEL = 6.3, 2.87
ESCALA_ENERGIA = "linear"            # "linear" (E_pot) ou "log" (E − E_final de cada etapa: convergência)
MOSTRAR_RAMPA = False                 # rampa nominal do .mdp no painel B
COR_BB, COR_SC = "#003f5c", "#bc5090"

# ── Gráfico ─────────────────────────────────────────────────────────────────
fig, (ax_a, ax_b, ax_c) = plt.subplots(3, 1, figsize=(LARGURA, 3 * ALTURA_PAINEL), layout="constrained")
mins = [e for e in ETAPAS if e["tipo"] == "minimizacao"]
dins = [e for e in ETAPAS if e["tipo"] == "dinamica"]
for i, e in enumerate(mins):
    g = df_series[df_series["etapa"] == e["rotulo"]]
    y = g["valor"] - g["valor"].min() if ESCALA_ENERGIA == "log" else g["valor"]
    ax_a.plot(g["eixo_acumulado"], y.where(y > 0) if ESCALA_ENERGIA == "log" else y,
              color=CORES[i % len(CORES)], lw=0.9, label=e["rotulo"])
    ax_a.axvline(g["eixo_acumulado"].iloc[-1], color="#969696", lw=0.5, ls="--")
if ESCALA_ENERGIA == "log":
    ax_a.set_yscale("log")
    ax_a.set_ylabel("E$_{pot}$ − E$_{pot,final}$ (kJ mol$^{-1}$)")
else:
    ax_a.ticklabel_format(axis="y", style="sci", scilimits=(0, 0), useMathText=True)   # ×10⁵ no topo do eixo
    ax_a.set_ylabel("E$_{pot}$ (kJ mol$^{-1}$)")
ax_a.set(xlabel="Passo acumulado de minimização")
ax_a.legend(ncol=len(mins), frameon=True, fontsize=TAMANHO_FONTE - 1)

for i, e in enumerate(dins):
    g = df_series[df_series["etapa"] == e["rotulo"]]
    ax_b.plot(g["tempo_acumulado_ns"], g["valor"], color=CORES[i % len(CORES)], lw=0.4, alpha=0.85, label=e["rotulo"])
    ax_b.axvline(g["tempo_acumulado_ns"].iloc[-1], color="#969696", lw=0.5, ls="--")
if MOSTRAR_RAMPA and ANNEALING and ETAPA_AQUECIMENTO in df_series["etapa"].values:
    inicio = df_series.loc[df_series["etapa"] == ETAPA_AQUECIMENTO, "eixo_acumulado"].iloc[0]
    fim = df_series["eixo_acumulado"][df_series["tipo"] == "dinamica"].max()
    tr = np.r_[np.array(ANNEALING["tempo_ps"]) + inicio, fim] / 1000.0
    Tr = np.r_[ANNEALING["temperatura_K"], ANNEALING["temperatura_K"][-1]]
    ax_b.plot(tr, Tr, color=COR_MEDIA, ls="--", lw=1.0, label="rampa (.mdp)")
ax_b.axhline(TEMPERATURA_ALVO, color=COR_REFERENCIA, ls=":", lw=0.8)
ax_b.set(xlabel="Tempo acumulado de dinâmica (ns)", ylabel="T (K)")
ax_b.legend(ncol=4, frameon=True, fontsize=TAMANHO_FONTE - 1)

posicoes, bordas, piso = np.arange(len(ETAPAS)), np.arange(len(ETAPAS) + 1) - 0.5, 1.0
ax_c.stairs(np.maximum(df_resumo["fc_bb"], piso), bordas, baseline=None, color=COR_BB, lw=1.2, label="cadeia principal")
ax_c.stairs(np.maximum(df_resumo["fc_sc"], piso), bordas, baseline=None, color=COR_SC, lw=1.2, ls="--", label="cadeias laterais")
for p in posicoes[(df_resumo["fc_bb"] == 0) & (df_resumo["fc_sc"] == 0)]:
    ax_c.axvspan(p - 0.5, p + 0.5, color=COR_REFERENCIA, alpha=0.10, lw=0)
ax_c.set_yscale("log")
ax_c.set_ylim(piso / 2, df_resumo[["fc_bb", "fc_sc"]].to_numpy().max() * 4)
ax_c.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(
    lambda v, _: "0" if np.isclose(v, piso) else (f"$10^{{{int(round(np.log10(v)))}}}$" if v > 0 else "")))
ax_c.set(xticks=posicoes, xlim=(-0.5, posicoes[-1] + 0.5), ylabel="k (kJ mol$^{-1}$ nm$^{-2}$)")
ax_c.set_xticklabels(df_resumo["etapa"], rotation=45, ha="right")
ax_c.legend(frameon=True, fontsize=TAMANHO_FONTE - 1)
for eixo, letra in zip((ax_a, ax_b, ax_c), "ABC"):
    eixo.set_title(letra, loc="left", fontweight="bold")
fig_etapas = fig
plt.show()

## 6 · Gráfico interativo (Plotly)

In [ ]:
# ── Layout ──────────────────────────────────────────────────────────────────
ALTURA_PX = 950                # pixels
LARGURA_PX = None              # None = largura da página
TEMPLATE = "simple_white"
TAMANHO_FONTE_PX = 13

# ── Gráfico ─────────────────────────────────────────────────────────────────
fig_i = make_subplots(rows=3, cols=1, vertical_spacing=0.13,
                      subplot_titles=("A — minimização", "B — dinâmica", "C — restrições posicionais"))
for i, e in enumerate(mins):
    g = df_series[df_series["etapa"] == e["rotulo"]]
    fig_i.add_trace(go.Scattergl(x=g["eixo_acumulado"], y=g["valor"], mode="lines", name=e["rotulo"],
                                 line=dict(color=CORES[i % len(CORES)])), row=1, col=1)
for i, e in enumerate(dins):
    g = df_series[df_series["etapa"] == e["rotulo"]]
    fig_i.add_trace(go.Scattergl(x=g["tempo_acumulado_ns"], y=g["valor"], mode="lines", name=e["rotulo"],
                                 line=dict(color=CORES[i % len(CORES)], width=0.8)), row=2, col=1)
fig_i.add_hline(y=TEMPERATURA_ALVO, line=dict(color=COR_REFERENCIA, dash="dot"), row=2, col=1)
for c, nome, cor, traco in (("fc_bb", "cadeia principal", COR_BB, "solid"), ("fc_sc", "cadeias laterais", COR_SC, "dash")):
    fig_i.add_trace(go.Scatter(x=posicoes, y=np.maximum(df_resumo[c], piso), mode="lines+markers", line_shape="hvh",
                               name=nome, line=dict(color=cor, dash=traco), customdata=df_resumo[c], text=df_resumo["etapa"],
                               hovertemplate="%{text}: k = %{customdata:.1f} kJ mol⁻¹ nm⁻²<extra></extra>"), row=3, col=1)
fig_i.update_xaxes(title_text="Passo acumulado", row=1, col=1)
fig_i.update_xaxes(title_text="Tempo acumulado (ns)", row=2, col=1)
fig_i.update_xaxes(tickvals=posicoes, ticktext=list(df_resumo["etapa"]), row=3, col=1)
fig_i.update_yaxes(title_text="E<sub>pot</sub> (kJ mol⁻¹)", exponentformat="power", row=1, col=1)   # sem o sufixo "k"
fig_i.update_yaxes(title_text="T (K)", row=2, col=1)
fig_i.update_yaxes(title_text="k (kJ mol⁻¹ nm⁻²)", type="log", dtick=1, row=3, col=1)
fig_i.update_layout(template=TEMPLATE, height=ALTURA_PX, width=LARGURA_PX, font=dict(family=FONTE, size=TAMANHO_FONTE_PX),
                    legend_title_text="Etapa")
fig_interativo = fig_i
fig_i.show()

## 7 · Exportação

Tudo vai para `DIR_SAIDA`, separado por formato:

```
{DIR_SAIDA}/
├── png/   600 dpi
├── pdf/   vetorial, fontes embutidas
├── svg/   vetorial, texto editável
├── html/  gráficos interativos (Plotly)
└── <prefixo>_dados.csv   ← CSV ÚNICO com todos os dados
```

O CSV é **um só**, em formato longo: a coluna `tabela` diz de que tabela vem
cada linha; as colunas que não se aplicam ficam vazias. As linhas iniciadas por
`#` registram data, hora, versões e parâmetros. Para ler:
`pd.read_csv(arquivo, comment="#")`.

| `tabela` | conteúdo |
|---|---|
| `serie` | energia (minimizações) e temperatura (dinâmicas), com eixo local e acumulado |
| `resumo` | por etapa: inicial, final, último quinto (média ± SEM), desvio do alvo, constantes, desfecho do `steep` |
| `arquivos` | arquivos lidos, termos e, nas minimizações, passos executados, Fmax e Epot finais |

In [ ]:
FORMATOS = ("png", "pdf", "svg")
DPI = 600
FUNDO_TRANSPARENTE = False
PLOTLY_JS = "cdn"                  # "cdn" = HTML leve; True = embute o plotly.js (abre offline)
EXPORTAR_PLOTLY_ESTATICO = False   # True = também PNG/PDF/SVG do Plotly (precisa do kaleido)

FIGURAS = {"etapas_equilibracao": fig_etapas}
INTERATIVOS = {"etapas_equilibracao": fig_interativo}
TABELAS = {"serie": df_series, "resumo": df_resumo, "arquivos": df_leitura}
PARAMETROS = {"temperatura_alvo": TEMPERATURA_ALVO, "annealing": ANNEALING, "fracao_final": FRACAO_FINAL}

# ── Figuras ─────────────────────────────────────────────────────────────────
for nome, fig in FIGURAS.items():
    for ext in FORMATOS:
        (DIR_SAIDA / ext).mkdir(parents=True, exist_ok=True)
        fig.savefig(DIR_SAIDA / ext / f"{nome}.{ext}", dpi=DPI, bbox_inches="tight",
                    transparent=FUNDO_TRANSPARENTE)
for nome, fig in INTERATIVOS.items():
    (DIR_SAIDA / "html").mkdir(parents=True, exist_ok=True)
    fig.write_html(DIR_SAIDA / "html" / f"{nome}.html", include_plotlyjs=PLOTLY_JS,
                   config={"displaylogo": False, "toImageButtonOptions": {"format": "svg"}})
    if EXPORTAR_PLOTLY_ESTATICO:
        for ext in FORMATOS:
            try:
                fig.write_image(DIR_SAIDA / ext / f"{nome}_interativo.{ext}", scale=3)
            except Exception as erro:
                print(f"[aviso] {nome}.{ext} do Plotly não exportado: {erro}")
                break

# ── CSV único ───────────────────────────────────────────────────────────────
DIR_SAIDA.mkdir(parents=True, exist_ok=True)
partes = []
for nome, tabela in TABELAS.items():
    t = tabela.copy()
    t.insert(0, "tabela", nome)
    partes.append(t)
dados = pd.concat(partes, ignore_index=True, sort=False)
arquivo_csv = DIR_SAIDA / f"{PREFIXO}_dados.csv"
with open(arquivo_csv, "w", encoding="utf-8", newline="") as f:
    f.write(f"# {TITULO}\n")
    f.write(f"# gerado em {INICIO:%Y-%m-%d %H:%M:%S} | sistema {SISTEMA} | réplicas {REPLICAS}\n")
    f.write("# versoes: " + "; ".join(f"{k} {v}" for k, v in VERSOES.items()) + "\n")
    f.write("# parametros: " + "; ".join(f"{k} = {v}" for k, v in PARAMETROS.items()) + "\n")
    f.write("# tabelas: " + ", ".join(f"{k} ({len(v)} linhas)" for k, v in TABELAS.items()) + "\n")
    dados.to_csv(f, index=False)

# ── O que foi gravado ───────────────────────────────────────────────────────
df_exportados = pd.DataFrame([{"pasta": str(p.parent.relative_to(DIR_SAIDA)), "arquivo": p.name,
                               "tamanho_kB": round(p.stat().st_size / 1024, 1)}
                              for p in sorted(DIR_SAIDA.rglob("*")) if p.is_file()])
print(f"Resultados em: {DIR_SAIDA.resolve()}")
print(f"Duração da execução: {datetime.now() - INICIO}")
df_exportados